# NBA history through recorded games

The four insights use the NBA dataset from Kaggle specified in Exercise 3 (`wyattowalsh/basketball`), querying the SQLite file `nba.sqlite`. The analysis periods and eligible records vary by question.

Run all cells from the repository root after preparing `data/nba.sqlite` as described in the [README](README.md).

In [ ]:
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
from IPython.display import display

from nba.data_loader import connect_database
from nba.insight_1 import home_win_trend
from nba.insight_1_viz import plot_home_win_trend
from nba.insight_2 import three_point_summary, three_point_trend
from nba.insight_2_viz import plot_three_point_trend
from nba.insight_3 import close_game_trend
from nba.insight_3_viz import plot_close_game_trend
from nba.insight_4_e import get_monthly_game_counts, plot_monthly_game_counts

## Insight 1 — Home wins across NBA seasons

**Research question.** How did the share of regular-season games won by the designated home team change from 1980–81 to 2022–23?

**Method.** Compare the final home and away scores for each recorded regular-season game and calculate the home win rate by season. Each eligible game contributes once. Games with missing identifiers, missing scores, or ties are excluded.

**Findings.** Across 42 recorded seasons, the designated home team won 28,502 of 47,027 eligible games (60.6%). The rate was 62.1% in 1980–81 and 58.0% in 2022–23. It reached 67.9% in both 1983–84 and 1987–88 and fell to 54.4% in 2020–21.

**Interpretation and limitations.** The rates describe wins by the designated home side. Neutral-site games, pandemic disruptions, scheduling and team strength affect how the results relate to home-court advantage. The dataset omits the 2012–13 season, and the number of recorded games varies across seasons.

[Source notebook](insight_1_home_wins.ipynb) · [SQL](queries/insight_1.sql)

In [ ]:
with closing(connect_database()) as connection:
    home_rates = home_win_trend(connection)

display(home_rates.iloc[[0, -1]].round({"home_win_pct": 1}))
with TemporaryDirectory() as folder:
    home_figure = plot_home_win_trend(home_rates, Path(folder) / "home_wins.png")
display(home_figure)
plt.close(home_figure)

## Insight 2 — The rise of the three-point shot

**Research question.** How did three-point shot volume and share change from 1981–82 to 2022–23 in the recorded regular seasons?

**Method.** Calculate average three-point attempts per team per game using records with attempts for both teams. For shot share, divide total three-point attempts by total field-goal attempts within the subset with both measures for both teams.

**Findings.** Average three-point attempts rose from 2.3 to 34.2 per team per game between 1981–82 and 2022–23. Three-pointers rose from 2.6% to 38.7% of field-goal attempts in the records with complete attempt data.

**Interpretation and limitations.** Three-point shots became more common in the available records. The first season has three-point attempt data for only 113 of 943 games, and the shot-share estimate uses 111 games. This limited coverage makes the early estimates less representative. The missing 2012–13 season appears as a gap in the source chart.

[Source notebook](insight_2.ipynb) · [SQL](queries/insight_2.sql)

In [ ]:
with closing(connect_database()) as connection:
    three_point_rates = three_point_trend(connection)

display(three_point_summary(three_point_rates))
with TemporaryDirectory() as folder:
    three_point_figure = plot_three_point_trend(
        three_point_rates, Path(folder) / "three_pointers.png"
    )
display(three_point_figure)
plt.close(three_point_figure)

## Insight 3 — How common are close NBA games?

**Research question.** Did the share of close NBA regular-season games change from 1980–81 to 2022–23?

**Method.** Use SQL to group eligible regular-season games by season and calculate the share decided by five points or fewer. Each eligible game has a valid ID, a recorded score for both teams and a winner.

**Findings.** The analysis covers 47,027 games across 42 available seasons. The share of close games fluctuated, reaching 35.21% in 1981–82. It was 32.77% in 1980–81 and 27.56% in 2022–23. In recent seasons it was generally around 25–28%.

**Interpretation and limitations.** Close games did not become consistently more common over this period. The missing 2012–13 season appears as a gap in the source chart. Final score margins can miss comebacks or games that stayed close until the final minutes.

[Source notebook](insight_3_close_games.ipynb) · [SQL](queries/insight_3.sql) · [PR #19](https://github.com/starries-creator/ps_2_ex3_database/pull/19)

In [ ]:
with closing(connect_database()) as connection:
    close_game_rates = close_game_trend(connection)

display(close_game_rates.iloc[[0, -1]])
with TemporaryDirectory() as folder:
    close_game_figure = plot_close_game_trend(
        close_game_rates, Path(folder) / "close_games.png"
    )
display(close_game_figure)
plt.close(close_game_figure)

## Insight 4 — When are NBA games most frequently played?

**Research question.** How does the number of recorded NBA games vary across calendar months?

**Method.** Extract the calendar month from `game_date` in the `game` table, count records in each month, and compare the counts in a bar chart.

**Findings.** The database contains 65,698 game records from 1 November 1946 to 12 June 2023. March has the most recorded games (11,269), followed by January (11,246) and December (10,587). July has the fewest (16). Records are concentrated from November to March, with much lower counts in summer.

**Interpretation and limitations.** The distribution is consistent with the NBA's seasonal calendar. The counts reflect the database's mix of game types and its historical coverage, so they describe the recorded games in this snapshot.

Member E’s write-up · [Source notebook](insight_4_E.ipynb) · [SQL](queries/insight_4.sql)

In [ ]:
with closing(connect_database()) as connection:
    monthly_counts = get_monthly_game_counts(connection)

display(monthly_counts)
monthly_figure, _ = plot_monthly_game_counts(monthly_counts)
display(monthly_figure)
plt.close(monthly_figure)